# CAIM Lab Session 4: Implementing search in the vector space model

In this session you will:

- Continue to work with the `arxiv` repository from last session
- Learn how to do atomic and compound search queries with ElasticSearch
- Build an inverted index for the `arxiv` repository from last session (should fit in main memory)
- Implement search in the vector space model and compare it with ElasticSearch built-in search mechanism
- Compare different implementations of search

## 1. Built-in search in ElasticSearch

ElasticSearch provides a search mechanism to make queries against a database. 
In the next code snippet you can find examples on how to do this with an atomic query (single term)
and a complex one with a so-called 'should' query (a type of OR) which admits weights in each term within the query.

In [ ]:
from elasticsearch import Elasticsearch
from pprint import pprint

client = Elasticsearch("http://localhost:9200", request_timeout=1000)

#### Atomic query

In [ ]:
# define query
atomic_query = {"match": {"text": "magic"}}

# search
response = client.search(index="arxiv", query=atomic_query, track_total_hits=True)

# show results
# Print the results
print(f"Found {response['hits']['total']['value']} documents.")
for hit in response["hits"]["hits"][:5]:
    print(
        f"id: {hit['_id']}, score: {hit['_score']:.2f}, path: {hit['_source']['path']}, text: {hit['_source'].get('text')}"
    )

#### Complex query with weights

In [ ]:
# define your query with set of weighted terms
weighted_terms = {
    "search": 0.5,
    "magic": 2.0,
}

# 3. build the 'should' clauses dynamically (behaves like an OR there are other options, too)
clauses = [
    {"match": {"text": {"query": term, "boost": weight}}}  # field to search over
    for term, weight in weighted_terms.items()
]

for clause_type in ['must', 'should']:
    print()
    print(f"query type with {clause_type} clauses")

    # construct the final bool query from set of weighted terms
    es_query = {"bool": {clause_type: clauses}}

    # execute the search
    response = client.search(index="arxiv", query=es_query, track_total_hits=True)

    # Print the results
    print(f"Found {response['hits']['total']['value']} documents.")
    for hit in response["hits"]["hits"][:10]:
        print(f"id: {hit['_id']}, score: {hit['_score']:.2f}, path: {hit['_source']['path']}, text: {hit['_source'].get('text')}")

## 2. Excruciatingly slow search

In class we have presented a _slow_ version of search that, given a search query $q$, loops over every document in the database
computing the cosine similarity between document and query. Once this is done, it sorts documents by their similarity w.r.t. $q$ and returns the top $r$
scoring ones. 

```
1. for each d in D:
    sim(d,q) = 0
    get vector representing d
    for each w in q:
        sim(d,q) += tf(d,w) * idf(w)
    normalize sim(d,q) by |d|*|q|
2. sort results by similarity
3. return top r docs
```

A possible implementation can be found below. 

__Remark:__ _It is important to note that there are certain elements in the implementation below that refer to my own
implementation, and that you should adapt to your own; in particular, the line_

```    weights = dict(normalize(tf_idf(s['_id'])))   # gets weights as a python dict of term -> weight ```

_obtains tf-idf weights through calling a function `tf_idf` that I have implemented that, given a docid, returns a list of pairs (term, weight); and `normalize` takes such a list a normalizes weights so that the corresponding vector has length 1. 
Obviously, you should adapt the code to your own implementations from previous sessions._


In [ ]:
import numpy as np
from collections import Counter

def tf_idf(doc_id):
    tv = client.termvectors(index="arxiv", id=doc_id, fields=["text"], term_statistics=True, field_statistics=True)
    if 'term_vectors' not in tv or 'text' not in tv['term_vectors']:
        return []
    terms = tv["term_vectors"]["text"]["terms"]
    total_docs = tv["term_vectors"]["text"]["field_statistics"]["doc_count"]
    weights = []
    for term, term_info in terms.items():
        tf = term_info["term_freq"]
        doc_freq = term_info.get("doc_freq", 1)
        idf = np.log(total_docs / (doc_freq + 1))
        tfidf = tf * idf
        weights.append((term, tfidf))
    return weights

def normalize(l1):
    if not l1:
        return []
    norma_l2 = np.sqrt(sum(w ** 2 for _, w in l1))
    
    if norma_l2 == 0:
        return l1    
    return [(term, w / norma_l2) for term, w in l1]

In [ ]:
from elasticsearch.helpers import scan
from pprint import pprint
from elasticsearch import Elasticsearch
import tqdm
import numpy as np


def preprocess_query_string(query_string, client, index_name, field_name):
    """
    given query string it outputs the list of preprocessed tokens from it
    using same analyzer (preprocessing pipeline) than the arxiv abstracts
    """

    # Use the analyze API on the specified index
    response = client.indices.analyze(
        index=index_name, field=field_name, text=query_string
    )

    # Extract just the token strings from the response
    preprocessed_terms = [token_info["token"] for token_info in response["tokens"]]

    # print(f"Original string: '{query_string}'")
    # print(f"Preprocessed terms: {preprocessed_terms}")
    return preprocessed_terms


client = Elasticsearch("http://localhost:9200", request_timeout=1000)

r = 1973  # only return r top docs
# query will be list of tokens, preprocessed like the indexed arxiv articles
query_str = "flower teacher magic science"
query_tokens = preprocess_query_string(
    query_string=query_str, client=client, index_name="arxiv", field_name="text"
)

print(f"Executing search of query string '{query_str}' with tokens {query_tokens} over documents on index 'arxiv'")
sims = dict()

l2query = np.sqrt(len(query_tokens))  # l2 of query assuming 0-1 vector representation

# get nr. of docs; just for the progress bar
ndocs = int(client.cat.count(index="arxiv", format="json")[0]["count"])

# scan through docs, compute cosine sim between query and each doc
for s in tqdm.tqdm(
    scan(client, index="arxiv", query={"query": {"match_all": {}}}), total=ndocs
):

    docid = s["_source"]["path"]  # use path as id
    weights = dict(
        normalize(tf_idf(s["_id"]))
    )  # gets weights as a python dict of term -> weight (see remark above)
    sims[docid] = 0.0
    for w in query_tokens:  # gets terms as a list
        if (
            w in weights
        ):  
            sims[docid] += weights[w]  # accumulates if w in current doc
    # normalize sim
    sims[docid] /= l2query

# now sort by cosine similarity
sorted_answer = sorted(sims.items(), key=lambda kv: kv[1], reverse=True)
pprint(sorted_answer[:r])

In [ ]:
nz = len([x for x, s in sorted_answer if s > 0])
total = len(sorted_answer)
print(
    f"There are {nz} docs with non-zero similarity out of {total}, i.e. {100.0*nz/total:.1f}%"
)

``` bash
curl -sS http://localhost:9200/_cat/indices?v
```

## 3. Your tasks

---

**Exercise 1:**  

Make sure you understand the algorithm for implementing search described in the lecture notes. Both slow and efficient versions. Describe
the number of operations you need to do in both slow and quick versions for the following toy example with a vocabulary of size 4 and four documents:

- $q = 0,1,1,0$

- document-term matrix:
<center>


|        | t1  | t2  | t3  | t4  |
|--------|-----|-----|-----|-----|
| **d1** | 1.2 | 0.0 | 0.0 | 0.0 |
| **d2** | 0.7 | 0.3 | 1.5 | 0.1 |
| **d3** | 0.0 | 0.0 | 0.0 | 0.7 |
| **d4** | 2.0 | 0.0 | 0.0 | 0.0 |

</center>

---


D = {d1, d2, d3, d4};
q = e1,e2,e3,e4

d = d1
sim(d1,q) = 0

sim(d1,q) += tf(d1,e1)*idf(e1)
sim(d1,q) += tf(d1,e2)*idf(e2)
sim(d1,q) += tf(d1,e3)*idf(e3)
sim(d1,q) += tf(d1,e4)*idf(e4)

normalize(d1,q) by |d1|*|q|

d = d2
sim(d2,q) = 0

sim(d2,q) += tf(d2,e1)*idf(e1)
sim(d2,q) += tf(d2,e2)*idf(e2)
sim(d2,q) += tf(d2,e3)*idf(e3)
sim(d2,q) += tf(d2,e4)*idf(e4)

normalize(d2,q) by |d2|*|q|

d = d3
sim(d3,q) = 0

sim(d3,q) += tf(d3,e1)*idf(e1)
sim(d3,q) += tf(d3,e2)*idf(e2)
sim(d3,q) += tf(d3,e3)*idf(e3)
sim(d3,q) += tf(d3,e4)*idf(e4)

normalize(d3,q) by |d3|*|q|

d = d4
sim(d4,q) = 0

sim(d4,q) += tf(d4,e1)*idf(e1)
sim(d4,q) += tf(d4,e2)*idf(e2)
sim(d4,q) += tf(d4,e3)*idf(e3)
sim(d4,q) += tf(d4,e4)*idf(e4)

normalize(d4,q) by |d4|*|q|

sort results by similarity

fast:
D = {d1, d2, d3, d4};
q = e1,e2,e3,e4

w = e1
L = {d1:1.2,d2:0.7,d4:2.0}


w = e2
L = {d2:0.3}
sim(d2,q) = 0
sim(d2, q) += tf(d2,e2) * idf(e2)

w = e3
L = {d2:1.5}
sim(d2, q) += tf(d2,e3) * idf(e3)

w = e4
L = {d2:0.1,d3:0.7}

normalize(d1,q) by |d1|*|q|
normalize(d2,q) by |d2|*|q|
normalize(d3,q) by |d3|*|q|
normalize(d4,q) by |d4|*|q|


sort results by simliarity

---

**Exercise 2:**

Implement the quick version; run both slow and quick versions and report times (as a reference, in my old laptop it takes around 5m20s to run the slow version in the code above). Make sure both versions return the same answer. Note that you will need to build an inverted index in order to implement the efficient version as explained in class; it may take time but this is done once for all queries, and can be done "off-line". Also, you could improve on the code by implementing the top-$r$ sort of the final answer using
the minheap tree as discussed in class. Python has a minheap built-in implementation called `heapq`.


In [ ]:
import heapq

def inverted_index(index):

    idx = dict()
    #idx = defaultdict(list)

    # get nr. of docs; just for the progress bar
    ndocs = int(client.cat.count(index=index, format="json")[0]["count"])

    # scan through docs, compute cosine sim between query and each doc
    for s in tqdm.tqdm(
        scan(client, index=index, query={"query": {"match_all": {}}}), total=ndocs
    ):
        docid = s["_source"]["path"]  # use path as id
        weights = dict(
            normalize(tf_idf(s["_id"]))
        )  # gets weights as a python dict of term -> weight (see remark above)

        for term, weight in weights.items():
            #idx[term].append((docid, weight))
            if term not in idx:
                idx[term] = []
            idx[term].append((docid, weight))

    return idx


In [ ]:
def quick_version(query_tokens, inv_idx, r):
    #query_tokens = preprocess_query_string(query_str, client, "arxiv", "text")
    norm = np.sqrt(len(query_tokens))
    sims = {} 
    for token in query_tokens:
        if token in inv_idx:  #Comprovem que el token estigui al index per si un cas
            for d, w in inv_idx[token]:
                if d not in sims:
                    sims[d] = 0.0
                sims[d] += w  #Ja hem calculat tf(d,w) * idf(w) en inverted_index
    for d in sims:
        sims[d] /= norm
    
    heap = []
    for d, s in sims.items():
        if len(heap) < r:
            heapq.heappush(heap, (s, d))
        elif s > heap[0][0]:
            heapq.heapreplace(heap, (s, d))
    
    sorted_query = sorted(heap, key=lambda x: x[0], reverse=True)
    return [(d, w) for w, d in sorted_query]

In [ ]:
import time
# Construir index invertit
start = time.time()
inv_idx = inverted_index("arxiv")
totalTime = time.time() - start

In [ ]:
start = time.time()
quick_results = quick_version(query_tokens, inv_idx, r)
quick_time = time.time() - start

nz = len([x for x, s in quick_results if s > 0])
total = len(quick_results)
print(
    f"There are {nz} docs with non-zero similarity out of {total}, i.e. {100.0*nz/total:.1f}%"
)

print(f"\nBúsqueda rápida ({quick_time:.4f}s):")
#pprint(quick_results[:r])


In [ ]:
def compare_searches(slow_results, quick_results, r):
    slow_top_r = slow_results[:r]
    
    match = True
    mismatches_by_score = []
    
    for i, ((slow_doc, slow_sim), (quick_doc, quick_sim)) in enumerate(zip(slow_top_r, quick_results)):
        if not np.isclose(slow_sim, quick_sim):
            print(f"Position {i} - SCORE MISMATCH:")
            print(f"  Slow:  {slow_sim:.10f}")
            print(f"  Quick: {quick_sim:.10f}")
            match = False
        elif slow_doc != quick_doc:
            mismatches_by_score.append((i, slow_sim, slow_doc, quick_doc))
    
    if mismatches_by_score:
        print(f"\n✓ Encontrados {len(mismatches_by_score)} documentos con similitud idéntica (empates):")
        for i, sim, slow_doc, quick_doc in mismatches_by_score:
            print(f"  Position {i} (sim={sim:.6f}): ambos son válidos")
            print(f"    - Slow:  {slow_doc}")
            print(f"    - Quick: {quick_doc}")
    
    if match:
        print(f"\n✓ Top-{r} resultados son IDÉNTICOS (incluyendo {len(mismatches_by_score)} empates resueltos)")
    
    return match

compare_searches(sorted_answer, quick_results, r)

---

**Exercise 3:**

Compare the results for a few sample queries that you get from your quick version and ElasticSearch search. Do you get similar results? Which is faster?

---

In [ ]:
# ============================================================================
# QUICK SEARCH (Vector Space Model with Cosine Similarity)
# ============================================================================

def quick_search(query_tokens, inv_idx, r=10):
    """Quick search using inverted index and minheap"""
    l2query = np.sqrt(len(query_tokens))
    sims = {}
    
    for token in query_tokens:
        if token in inv_idx:
            for d, w in inv_idx[token]:
                if d not in sims:
                    sims[d] = 0.0
                sims[d] += w
    
    # Normalize
    for d in sims:
        sims[d] /= l2query
    
    # Use heapq.nlargest for efficiency: O(n log r) instead of O(n log n)
    top_r = heapq.nlargest(r, sims.items(), key=lambda x: (x[1], x[0]))
    
    return top_r

# ============================================================================
# SLOW SEARCH (Vector Space Model - ALL documents)
# ============================================================================

def slow_search(query_tokens, index_name, r=10):
    """Slow search scanning all documents"""
    sims = dict()
    l2query = np.sqrt(len(query_tokens))
    ndocs = int(client.cat.count(index=index_name, format="json")[0]["count"])
    
    for s in tqdm.tqdm(
        scan(client, index=index_name, query={"query": {"match_all": {}}}), 
        total=ndocs,
        desc="Scanning all docs"
    ):
        docid = s["_source"]["path"]
        weights = dict(normalize(tf_idf(s["_id"])))
        
        sims[docid] = 0.0
        for w in query_tokens:
            if w in weights:
                sims[docid] += weights[w]
        sims[docid] /= l2query
    
    sorted_results = sorted(sims.items(), key=lambda kv: kv[1], reverse=True)
    return sorted_results[:r]

# ============================================================================
# ELASTICSEARCH SEARCH (TF-IDF)
# ============================================================================

def elasticsearch_search_tfidf(query_str, client, index_name, r=10):
    """Elasticsearch search using TF-IDF scoring"""
    start_time = time.time()
    
    query = {"match": {"text": query_str}}
    response = client.search(index=index_name, query=query, size=r)
    
    elapsed_time = time.time() - start_time
    
    results = []
    for hit in response["hits"]["hits"]:
        docid = hit["_source"]["path"]
        score = hit["_score"]
        results.append((docid, score))
    
    return results, elapsed_time

# ============================================================================
# COMPARE METHODS
# ============================================================================

def compare_all_methods(query_str, inv_idx, client, r=50):
    """Compare slow, quick, and elasticsearch methods"""
    
    print(f"\n{'='*90}")
    print(f"QUERY: '{query_str}'")
    print(f"{'='*90}\n")
    
    query_tokens = preprocess_query_string(query_str, client, "arxiv", "text")
    print(f"Preprocessed tokens: {query_tokens}\n")
    
    # 1. SLOW SEARCH
    print("1️⃣  SLOW SEARCH (Vector Space - scans ALL documents)")
    print("-" * 90)
    start_slow = time.time()
    slow_results = slow_search(query_tokens, "arxiv", r)
    slow_time = time.time() - start_slow
    
    print(f"⏱️  Time: {slow_time:.2f}s\n")
    print("Top 10 results:")
    for i, (doc, score) in enumerate(slow_results[:10], 1):
        print(f"  {i:2d}. {doc}: {score:.6f}")
    
    # 2. QUICK SEARCH
    print(f"\n2️⃣  QUICK SEARCH (Inverted Index - processes only relevant docs)")
    print("-" * 90)
    start_quick = time.time()
    quick_results = quick_search(query_tokens, inv_idx, r)
    quick_time = time.time() - start_quick
    
    print(f"⏱️  Time: {quick_time:.4f}s\n")
    print("Top 10 results:")
    for i, (doc, score) in enumerate(quick_results[:10], 1):
        print(f"  {i:2d}. {doc}: {score:.6f}")
    
    # 3. ELASTICSEARCH SEARCH
    print(f"\n3️⃣  ELASTICSEARCH SEARCH (Built-in TF-IDF)")
    print("-" * 90)
    es_results, es_time = elasticsearch_search_tfidf(query_str, client, "arxiv", r)
    
    print(f"⏱️  Time: {es_time:.4f}s\n")
    print("Top 10 results:")
    for i, (doc, score) in enumerate(es_results[:10], 1):
        print(f"  {i:2d}. {doc}: {score:.6f}")
    
    # 4. PERFORMANCE COMPARISON
    print(f"\n{'='*90}")
    print("⚡ PERFORMANCE COMPARISON")
    print(f"{'='*90}")
    print(f"Slow search:          {slow_time:10.2f}s  (baseline)")
    print(f"Quick search:         {quick_time:10.4f}s  ({slow_time/quick_time:7.0f}x faster than slow)")
    print(f"Elasticsearch:        {es_time:10.4f}s  ({slow_time/es_time:7.0f}x faster than slow)")
    
    # 5. RESULT SIMILARITY
    print(f"\n{'='*90}")
    print("📊 RESULT SIMILARITY ANALYSIS (Top 10)")
    print(f"{'='*90}")
    
    slow_docs = {doc for doc, _ in slow_results[:10]}
    quick_docs = {doc for doc, _ in quick_results[:10]}
    es_docs = {doc for doc, _ in es_results[:10]}
    
    slow_quick = len(slow_docs & quick_docs)
    slow_es = len(slow_docs & es_docs)
    quick_es = len(quick_docs & es_docs)
    
    print(f"Slow vs Quick:        {slow_quick}/10 documents match ({slow_quick*10}%)")
    print(f"Slow vs Elasticsearch:{slow_es}/10 documents match ({slow_es*10}%)")
    print(f"Quick vs Elasticsearch:{quick_es}/10 documents match ({quick_es*10}%)")
    
    # 6. SCORE COMPARISON
    print(f"\n{'='*90}")
    print("📈 SCORE COMPARISON (Top 5)")
    print(f"{'='*90}")
    
    print(f"\nSlow:          Quick:         Elasticsearch:")
    for i in range(min(5, len(slow_results), len(quick_results), len(es_results))):
        slow_score = slow_results[i][1] if i < len(slow_results) else 0
        quick_score = quick_results[i][1] if i < len(quick_results) else 0
        es_score = es_results[i][1] if i < len(es_results) else 0
        
        print(f"{slow_score:.6f}      {quick_score:.6f}         {es_score:.6f}")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":
    # Build inverted index (one time)
    print("="*90)
    print("INITIALIZING...")
    print("="*90)
    inv_idx = inverted_index("arxiv")
    
    # Test queries
    test_queries = [
        "searching magic",
        "quantum computing",
        "machine learning",
    ]
    
    for query in test_queries:
        compare_all_methods(query, inv_idx, client, r=50)
    
    print(f"\n{'='*90}")
    print("✓ Comparison complete!")
    print(f"{'='*90}")


## 4. Rules of delivery

- To be solved in _pairs_.

- No plagiarism; don't discuss your work with other teams. You can ask for help to others for simple things, such as recalling a python instruction or module, but nothing too specific to the session.

- If you feel you are spending much more time than the rest of the classmates, ask us for help. Questions can be asked either in person or by email, and you'll never be penalized by asking questions, no matter how stupid they look in retrospect.

- Write a short report listing the solutions to the exercises proposed. Include things like the important parts of your implementation (data structures used for representing objects, algorithms used, etc). You are welcome to add conclusions and findings that depart from what we asked you to do. We encourage you to discuss the difficulties you find; this lets us give you help and also improve the lab session for future editions.

- Turn the report to PDF. Make sure it has your names, date, and title. Include your code in your submission.

- Submit your work through the [raco](http://www.fib.upc.edu/en/serveis/raco.html); see date at the raco's submissions page.